In [13]:
import pandas as pd
from jiwer import (
    wer, cer, mer, wil,
    process_words,
    Compose, ToLowerCase, RemovePunctuation,
    RemoveMultipleSpaces, Strip,
)

# ── Text normalisation applied before scoring ──────────────────────────────
TRANSFORM = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    RemoveMultipleSpaces(),
    Strip(),
])


def score(references: list[str], hypotheses: list[str]) -> dict:
    refs_norm = [TRANSFORM(r) for r in references]
    hyps_norm = [TRANSFORM(h) for h in hypotheses]
    # Single pass for word-level stats
    pw = process_words(refs_norm, hyps_norm)
    total = pw.hits + pw.substitutions + pw.deletions
    return {
        "WER": round(wer(refs_norm, hyps_norm) * 100, 2),
        "CER": round(cer(refs_norm, hyps_norm) * 100, 2),
        "MER": round(mer(refs_norm, hyps_norm) * 100, 2),
        "WIL": round(wil(refs_norm, hyps_norm) * 100, 2),
        "SUB": pw.substitutions,
        "DEL": pw.deletions,
        "INS": pw.insertions,
        "TOT": total,
    }


def per_utterance_wer(df: pd.DataFrame,
                      ref_col: str = "transcript",
                      hyp_col: str = "hypothesis") -> pd.DataFrame:
    df = df.copy()
    df["utt_wer"] = df.apply(
        lambda row: round(
            wer(TRANSFORM(str(row[ref_col])), TRANSFORM(str(row[hyp_col]))) * 100, 2
        ),
        axis=1,
    )
    return df


def print_metrics(label: str, metrics: dict):
    print(f"\n{'='*40}")
    print(f"  {label}")
    print(f"{'='*40}")
    for k in ("WER", "CER", "MER", "WIL"):
        print(f"  {k:<5}: {metrics[k]:>6.2f}%")
    print(f"  --- word counts ---")
    print(f"  SUB={metrics['SUB']}  DEL={metrics['DEL']}  INS={metrics['INS']}  TOT={metrics['TOT']}")


def show_examples(df: pd.DataFrame, n: int = 5, worst: bool = False):
    """Show best or worst utterances by per-utterance WER."""
    label = "Worst" if worst else "Best"
    subset = df.nlargest(n, "utt_wer") if worst else df.nsmallest(n, "utt_wer")
    print(f"\n--- {label} {n} utterances ---")
    for _, row in subset.iterrows():
        print(f"  WER={row['utt_wer']:5.1f}%")
        print(f"    REF: {row['transcript']}")
        print(f"    HYP: {row['hypothesis']}")


def evaluate(csv_path: str, label: str = "Baseline",
             n_examples: int = 5) -> tuple[pd.DataFrame, dict]:
    df = pd.read_csv(csv_path)
    metrics = score(df["transcript"].tolist(), df["hypothesis"].tolist())
    print_metrics(label, metrics)
    df = per_utterance_wer(df)
    show_examples(df, n=n_examples, worst=False)
    show_examples(df, n=n_examples, worst=True)
    return df, metrics


In [14]:
BASELINE_CSV = "/home/muthuajay/Documents/GitHub/STT/Deepseek_moe/baseline_results.csv"

baseline_df, baseline_metrics = evaluate(
        BASELINE_CSV, "Baseline (whisper-large-v3)", n_examples=5
    )


  Baseline (whisper-large-v3)
  WER  :  10.20%
  CER  :   2.50%
  MER  :   9.99%
  WIL  :  15.71%
  --- word counts ---
  SUB=265  DEL=83  INS=88  TOT=4276

--- Best 5 utterances ---
  WER=  0.0%
    REF: Consequently, the designated expert will intend to assemble vastly different types of knowledge in its parameters, which are hard to utilize simultaneously. 2 Knowledge Redundancy: tokens assigned to different experts may require common knowledge.
    HYP: Consequently, the designated expert will intend to assemble vastly different types of knowledge in its parameters, which are hard to utilize simultaneously. 2. Knowledge redundancy. Tokens assigned to different experts may require common knowledge.
  WER=  0.0%
    REF: As a result, multiple experts may converge in acquiring shared knowledge in their respective parameters, thereby leading to redundancy in expert parameters.
    HYP: As a result, multiple experts may converge in acquiring shared knowledge in their respective paramet

In [3]:
show_examples(baseline_df, n=10, worst=True)


--- Worst 10 utterances ---
  WER= 76.5%
    REF: DeepSeekMoE: Towards Ultimate Expert Specialization in Mixture of-Experts Language Models Damai Dai1,2, Chengqi Deng1, Chenggang Zhao1,3, R.X.
    HYP: Kip Sik Mo, Towards Ultimate Expert Specialization in Mixture of Experts Language Models, Demai Dai 1-2, Chengchi Deng 1, Chenggang Zhao 1-3, Rx.
  WER= 50.0%
    REF: RACE: large scale reading comprehension dataset from examinations.
    HYP: Race large-scale reading comprehension data set from examinations.
  WER= 50.0%
    REF: Deepspeed moe: Advancing mixture of-experts inference and training to power next generation AI scale.
    HYP: Deep Speed Mo, advancing mixture of experts' inference and training to power next-generation AI scale.
  WER= 44.4%
    REF: Subfigure b illustrates the fine grained expert segmentation strategy.
    HYP: So figure B illustrates the fine-grained expert segmentation strategy.
  WER= 40.0%
    REF: For the preliminary study of DeepSeekMoE 145B, we emplo

In [8]:
import os
from tqdm import tqdm
from faster_whisper import WhisperModel


WHISPER_MODEL_SIZE   = os.environ.get("STT_MODEL",   "large-v3")
WHISPER_COMPUTE_TYPE = os.environ.get("STT_COMPUTE", "float16")


WHISPER_INITIAL_PROMPT = os.environ.get(
    "STT_PROMPT",
    (
        "DeepSeekMoE, GShard, ZeRO, DeepSpeed, MoE, mixture-of-experts, "
        "fine-grained expert segmentation, shared expert isolation, "
        "expert specialization, Transformer, hidden dimension, "
        "warmup scheduler, activated parameters, language model."
    ),
)



In [9]:
def retranscribe_with_prompt(df: pd.DataFrame,
                              prompt: str = WHISPER_INITIAL_PROMPT) -> list[str]:
    """Re-run Whisper on the original audio with a domain initial_prompt."""
    

    print("[improvement] Loading model for re-transcription …")
    model = WhisperModel(WHISPER_MODEL_SIZE, compute_type=WHISPER_COMPUTE_TYPE)
    print(f"[improvement] Re-transcribing {len(df)} files with initial_prompt …")
    print(f"[improvement] Prompt: {prompt[:80]}…")

    hypotheses = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Re-transcribing"):
        segments, _ = model.transcribe(
            row["audio_path"],
            language="en",
            beam_size=5,
            vad_filter=True,
            initial_prompt=prompt,
        )
        hyp = " ".join(seg.text.strip() for seg in segments).strip()
        hypotheses.append(hyp)
    return hypotheses

In [10]:
baseline_df["hypothesis"] = retranscribe_with_prompt(baseline_df)

[improvement] Loading model for re-transcription …
[improvement] Re-transcribing 200 files with initial_prompt …
[improvement] Prompt: DeepSeekMoE, GShard, ZeRO, DeepSpeed, MoE, mixture-of-experts, fine-grained expe…


Re-transcribing: 100%|██████████| 200/200 [01:15<00:00,  2.65it/s]


,audio_path,transcript,style,word_count,char_len,hypothesis,runtime_s,utt_wer
0,/home/muthuajay/Documents/GitHub/STT/synthetic...,DeepSeekMoE: Towards Ultimate Expert Specializ...,neutral,17,142,"Kip Sik Mo, Towards Ultimate Expert Specializa...",0.504,76.47
1,/home/muthuajay/Documents/GitHub/STT/synthetic...,"However, conventional MoE architectures like G...",[emphasis],28,207,"However, conventional MAUI architectures like ...",0.420,10.71
2,/home/muthuajay/Documents/GitHub/STT/synthetic...,"In response, we propose the DeepSeekMoE archit...",[low voice],11,92,"In response, we propose the DeepSeq Mo archite...",0.251,18.18
3,/home/muthuajay/Documents/GitHub/STT/synthetic...,Starting from a modest scale with 2B parameter...,[excited tone],26,182,Starting from a modest scale with 2B parameter...,0.403,11.54
4,/home/muthuajay/Documents/GitHub/STT/synthetic...,"In addition, DeepSeekMoE 2B nearly approaches ...",[professional broadcast tone],27,169,"In addition, DeepSeek MO 2B nearly approaches ...",0.347,7.41
...,...,...,...,...,...,...,...,...
195,/home/muthuajay/Documents/GitHub/STT/synthetic...,RACE: large scale reading comprehension datase...,neutral,8,66,Race large-scale reading comprehension data se...,0.227,50.00
196,/home/muthuajay/Documents/GitHub/STT/synthetic...,Efficient large scale language model training ...,[emphasis],12,80,Efficient large-scale language model training ...,0.264,16.67
197,/home/muthuajay/Documents/GitHub/STT/synthetic...,Zero: memory optimizations toward training tri...,[low voice],8,69,0. Memory optimizations toward training trilli...,0.244,37.50
198,/home/muthuajay/Documents/GitHub/STT/synthetic...,Deepspeed moe: Advancing mixture of-experts in...,[excited tone],14,101,"Deep Speed Mo, advancing mixture of experts' i...",0.259,50.00


In [11]:
show_examples(baseline_df, n=10, worst=True)


--- Worst 10 utterances ---
  WER= 76.5%
    REF: DeepSeekMoE: Towards Ultimate Expert Specialization in Mixture of-Experts Language Models Damai Dai1,2, Chengqi Deng1, Chenggang Zhao1,3, R.X.
    HYP: DeepSeekMoE, towards ultimate expert specialization in mixture-of-experts language models, Demai Dai 1.2, Chengchi Deng 1, Chenggang Zhao 1.3, Rx.
  WER= 50.0%
    REF: RACE: large scale reading comprehension dataset from examinations.
    HYP: RACE large-scale reading comprehension data set from examinations.
  WER= 50.0%
    REF: Deepspeed moe: Advancing mixture of-experts inference and training to power next generation AI scale.
    HYP: DeepSpeed MoE, advancing mixture-of-experts inference and training to power next-generation AI scale.
  WER= 44.4%
    REF: Subfigure b illustrates the fine grained expert segmentation strategy.
    HYP: So figure B illustrates the fine-grained expert segmentation strategy.
  WER= 40.0%
    REF: For the preliminary study of DeepSeekMoE 145B, we emplo

In [16]:
import os
import re
import argparse
import pandas as pd
from jiwer import (
    wer, cer, mer, wil,
    process_words,
    Compose, ToLowerCase, RemovePunctuation,
    RemoveMultipleSpaces, Strip,
)


def _normalize(text: str) -> str:
    """Pre-processing before jiwer TRANSFORM: collapse intra-word hyphens."""
    # "fine-grained" → "fine grained", "mixture-of-experts" → "mixture of experts"
    text = re.sub(r"(?<=\w)-(?=\w)", " ", text)
    return text


# ── Text normalisation applied before scoring ──────────────────────────────
_JIWER_TRANSFORM = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    RemoveMultipleSpaces(),
    Strip(),
])


def TRANSFORM(text: str) -> str:
    return _JIWER_TRANSFORM(_normalize(text))


def score(references: list[str], hypotheses: list[str]) -> dict:
    refs_norm = [TRANSFORM(r) for r in references]
    hyps_norm = [TRANSFORM(h) for h in hypotheses]
    # Single pass for word-level stats
    pw = process_words(refs_norm, hyps_norm)
    total = pw.hits + pw.substitutions + pw.deletions
    return {
        "WER": round(wer(refs_norm, hyps_norm) * 100, 2),
        "CER": round(cer(refs_norm, hyps_norm) * 100, 2),
        "MER": round(mer(refs_norm, hyps_norm) * 100, 2),
        "WIL": round(wil(refs_norm, hyps_norm) * 100, 2),
        "SUB": pw.substitutions,
        "DEL": pw.deletions,
        "INS": pw.insertions,
        "TOT": total,
    }


def per_utterance_wer(df: pd.DataFrame,
                      ref_col: str = "transcript",
                      hyp_col: str = "hypothesis") -> pd.DataFrame:
    df = df.copy()
    df["utt_wer"] = df.apply(
        lambda row: round(
            wer(TRANSFORM(str(row[ref_col])), TRANSFORM(str(row[hyp_col]))) * 100, 2
        ),
        axis=1,
    )
    return df


def print_metrics(label: str, metrics: dict):
    print(f"\n{'='*40}")
    print(f"  {label}")
    print(f"{'='*40}")
    for k in ("WER", "CER", "MER", "WIL"):
        print(f"  {k:<5}: {metrics[k]:>6.2f}%")
    print(f"  --- word counts ---")
    print(f"  SUB={metrics['SUB']}  DEL={metrics['DEL']}  INS={metrics['INS']}  TOT={metrics['TOT']}")


def show_examples(df: pd.DataFrame, n: int = 5, worst: bool = False):
    """Show best or worst utterances by per-utterance WER."""
    label = "Worst" if worst else "Best"
    subset = df.nlargest(n, "utt_wer") if worst else df.nsmallest(n, "utt_wer")
    print(f"\n--- {label} {n} utterances ---")
    for _, row in subset.iterrows():
        print(f"  WER={row['utt_wer']:5.1f}%")
        print(f"    REF: {row['transcript']}")
        print(f"    HYP: {row['hypothesis']}")


def evaluate(csv_path: str, label: str = "Baseline",
             n_examples: int = 5) -> tuple[pd.DataFrame, dict]:
    df = pd.read_csv(csv_path)
    metrics = score(df["transcript"].tolist(), df["hypothesis"].tolist())
    print_metrics(label, metrics)
    df = per_utterance_wer(df)
    show_examples(df, n=n_examples, worst=False)
    show_examples(df, n=n_examples, worst=True)
    return df, metrics

baseline_df, baseline_metrics = evaluate(
        BASELINE_CSV, "Baseline (whisper-large-v3)", n_examples=5
    )




  Baseline (whisper-large-v3)
  WER  :   8.87%
  CER  :   2.38%
  MER  :   8.62%
  WIL  :  13.58%
  --- word counts ---
  SUB=230  DEL=27  INS=123  TOT=4283

--- Best 5 utterances ---
  WER=  0.0%
    REF: Conventional MoE architectures substitute the Feed Forward Networks FFNs in a Transformer with MoE layers.
    HYP: Conventional MOE architectures substitute the feed-forward network's FFNs in a transformer with MOE layers.
  WER=  0.0%
    REF: Consequently, the designated expert will intend to assemble vastly different types of knowledge in its parameters, which are hard to utilize simultaneously. 2 Knowledge Redundancy: tokens assigned to different experts may require common knowledge.
    HYP: Consequently, the designated expert will intend to assemble vastly different types of knowledge in its parameters, which are hard to utilize simultaneously. 2. Knowledge redundancy. Tokens assigned to different experts may require common knowledge.
  WER=  0.0%
    REF: As a result, multip

In [17]:
baseline_df["hypothesis"] = retranscribe_with_prompt(baseline_df)

[improvement] Loading model for re-transcription …
[improvement] Re-transcribing 200 files with initial_prompt …
[improvement] Prompt: DeepSeekMoE, GShard, ZeRO, DeepSpeed, MoE, mixture-of-experts, fine-grained expe…


Re-transcribing: 100%|██████████| 200/200 [01:11<00:00,  2.82it/s]


In [18]:
show_examples(baseline_df, n=5, worst=True)


--- Worst 5 utterances ---
  WER= 72.2%
    REF: DeepSeekMoE: Towards Ultimate Expert Specialization in Mixture of-Experts Language Models Damai Dai1,2, Chengqi Deng1, Chenggang Zhao1,3, R.X.
    HYP: DeepSeekMoE, towards ultimate expert specialization in mixture-of-experts language models, Demai Dai 1.2, Chengchi Deng 1, Chenggang Zhao 1.3, Rx.
  WER= 39.1%
    REF: In contrast, GShard1.5 exhibits greater redundancy among its expert parameters, so it can buffer the performance drop when top routed experts are disabled.
    HYP: In contrast, GShard 1.5 exhibits greater redundancy among its expert parameters, so it can buffer the performance drop.
  WER= 35.7%
    REF: With only 4 routed experts activated, DeepSeekMoE achieves a Pile loss comparable with GShard.
    HYP: With only four routed experts activated, DeepSeekMoE achieves a pile loss comparable with GShard.
  WER= 33.3%
    REF: DeepSeekMoE achieves comparable performance with a GShard model containing 1.5 times expert parame